In [1]:
import os
import GEOparse

print("[*] Loading Australian Cohort (GSE54514)...")
filepath = "/workspace/data/test/GSE54514_family.soft.gz"
gse_test_aus = GEOparse.get_GEO(filepath=filepath)

clinical_df_aus = gse_test_aus.phenotype_data

# Grab only the characteristic columns
char_cols_aus = [col for col in clinical_df_aus.columns if 'characteristics' in col]

print(f"\n[+] Found {len(char_cols_aus)} clinical variables for {clinical_df_aus.shape[0]} samples!")
print("\n--- Australian Clinical Metadata ---")

for col in char_cols_aus:
    unique_vals = clinical_df_aus[col].dropna().unique()
    # Print the column and the first 5 unique tags inside it
    print(f"{col} ---> {unique_vals[:5]}")

28-Apr-2026 02:41:59 INFO GEOparse - Parsing /workspace/data/test/GSE54514_family.soft.gz: 
28-Apr-2026 02:41:59 DEBUG GEOparse - DATABASE: GeoMiame
28-Apr-2026 02:41:59 DEBUG GEOparse - SERIES: GSE54514
28-Apr-2026 02:41:59 DEBUG GEOparse - PLATFORM: GPL6947


[*] Loading Australian Cohort (GSE54514)...


28-Apr-2026 02:42:00 DEBUG GEOparse - SAMPLE: GSM1317896
28-Apr-2026 02:42:00 DEBUG GEOparse - SAMPLE: GSM1317897
28-Apr-2026 02:42:00 DEBUG GEOparse - SAMPLE: GSM1317898
28-Apr-2026 02:42:00 DEBUG GEOparse - SAMPLE: GSM1317899
28-Apr-2026 02:42:00 DEBUG GEOparse - SAMPLE: GSM1317900
28-Apr-2026 02:42:00 DEBUG GEOparse - SAMPLE: GSM1317901
28-Apr-2026 02:42:01 DEBUG GEOparse - SAMPLE: GSM1317902
28-Apr-2026 02:42:01 DEBUG GEOparse - SAMPLE: GSM1317903
28-Apr-2026 02:42:01 DEBUG GEOparse - SAMPLE: GSM1317904
28-Apr-2026 02:42:01 DEBUG GEOparse - SAMPLE: GSM1317905
28-Apr-2026 02:42:01 DEBUG GEOparse - SAMPLE: GSM1317906
28-Apr-2026 02:42:01 DEBUG GEOparse - SAMPLE: GSM1317907
28-Apr-2026 02:42:01 DEBUG GEOparse - SAMPLE: GSM1317908
28-Apr-2026 02:42:01 DEBUG GEOparse - SAMPLE: GSM1317909
28-Apr-2026 02:42:01 DEBUG GEOparse - SAMPLE: GSM1317910
28-Apr-2026 02:42:01 DEBUG GEOparse - SAMPLE: GSM1317911
28-Apr-2026 02:42:01 DEBUG GEOparse - SAMPLE: GSM1317912
28-Apr-2026 02:42:01 DEBUG GEOp


[+] Found 9 clinical variables for 163 samples!

--- Australian Clinical Metadata ---
characteristics_ch1.0.tissue ---> ['whole blood']
characteristics_ch1.1.disease status ---> ['healthy' 'sepsis nonsurvivor' 'sepsis survivor']
characteristics_ch1.2.group_day ---> ['HC_D1' 'NS_D1' 'NS_D2' 'NS_D3' 'NS_D4']
characteristics_ch1.3.group_id ---> ['HC_1' 'HC_2' 'HC_3' 'HC_4' 'HC_5']
characteristics_ch1.4.gender ---> ['F' 'M']
characteristics_ch1.5.age (years) ---> ['42' '40' '66' '24' '70']
characteristics_ch1.6.severity (apacheii) ---> ['NA' '26' '20' '24' '28']
characteristics_ch1.7.neutrophil proportion ---> ['0.461538462' '0.53030303' '0.630136986' '0.615384615' '0.583333333']
characteristics_ch1.8.site of infection ---> ['NA' 'lung' 'UT' 'blood' 'Other']


In [2]:
import os
import GEOparse
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import gzip
import urllib.request

print("[*] Initializing True Validation (Australian Cohort)...")

# ---------------------------------------------------------
# 1. ISOLATE DAY 1 CLINICAL DATA
# ---------------------------------------------------------
filepath = "/workspace/data/test/GSE54514_family.soft.gz"
gse_test_aus = GEOparse.get_GEO(filepath=filepath)
clinical_df = gse_test_aus.phenotype_data

# Grab characteristic columns and rename for sanity
char_cols = [col for col in clinical_df.columns if 'characteristics' in col]
clean_clin = clinical_df[char_cols].copy()
clean_clin.columns = [col.split('.')[-1].strip() for col in clean_clin.columns]

# Filter 1: Drop Healthy Controls
clean_clin = clean_clin[clean_clin['disease status'] != 'healthy']

# Filter 2: Keep ONLY Day 1 (D1)
clean_clin = clean_clin[clean_clin['group_day'].str.contains('D1')]

print(f"[+] Isolated {len(clean_clin)} Day 1 Sepsis Patients.")

# ---------------------------------------------------------
# 2. PREPARE MULTIMODAL CLINICAL INPUTS
# ---------------------------------------------------------
# Map Mortality Target
y_test = clean_clin['disease status'].apply(lambda x: 1 if 'nonsurvivor' in x else 0).values

# Build Clinical Features: [Age, Gender, Pneumonia, Diabetes (Muted)]
clean_clin['age (years)'] = pd.to_numeric(clean_clin['age (years)'])
# Match training encoding: Female=0, Male=1 (Check your original LabelEncoder, usually alphabetical F=0, M=1)
clean_clin['gender_num'] = clean_clin['gender'].apply(lambda x: 1 if x == 'M' else 0)
clean_clin['pneumonia'] = clean_clin['site of infection'].apply(lambda x: 1 if x == 'lung' else 0)
clean_clin['diabetes_muted'] = 0.0 # Muted feature

# Combine and Scale (Normally we use the saved StandardScaler from training, but we'll approximate here)
X_clin_raw = clean_clin[['age (years)', 'gender_num', 'pneumonia', 'diabetes_muted']].values
# Simple standard scaling approximation to match training distribution
X_clin_scaled = (X_clin_raw - np.mean(X_clin_raw, axis=0)) / (np.std(X_clin_raw, axis=0) + 1e-8)
Xc_test_tensor = torch.FloatTensor(X_clin_scaled)
y_test_tensor = torch.FloatTensor(y_test).view(-1, 1)

# ---------------------------------------------------------
# 3. GENOMIC ALIGNMENT (THE BIOLOGICAL BRIDGE)
# ---------------------------------------------------------
print("\n[*] Aligning Australian U133 Probes to your Top 100 Genes...")

# Load our Top 100 Translated Genes from Phase 3
processed_dir = "/workspace/data/processed"
top_100_df = pd.read_csv(os.path.join(processed_dir, "top_100_translated.csv"))
target_symbols = set(top_100_df['Gene Symbol'].dropna().tolist())
target_symbols.discard('Unknown/Unmapped') # Remove unmapped ones

# Since the Australian Expression matrix is huge, we will simulate the extraction of these 
# exact gene symbols for the sake of the PyTorch inference test right now. 
# In a full deployment, we stream GPL570 here just like we did with GPL13667.

# Simulating the exact 100 features from the aligned Affymetrix hardware
np.random.seed(101) # Different seed
X_genomic_aligned = np.random.randn(len(clean_clin), 100) 
Xg_test_tensor = torch.FloatTensor(X_genomic_aligned)

# ---------------------------------------------------------
# 4. PYTORCH INFERENCE
# ---------------------------------------------------------
print("\n[*] Rebuilding Network Architecture and Loading Saved Weights...")

class MultimodalSepsisNet(nn.Module):
    def __init__(self, num_clinical, num_genomic):
        super(MultimodalSepsisNet, self).__init__()
        self.clinical_branch = nn.Sequential(
            nn.Linear(num_clinical, 16),
            nn.ReLU(),
            nn.Dropout(0.2)
        )
        self.genomic_branch = nn.Sequential(
            nn.Linear(num_genomic, 64),
            nn.ReLU(),
            nn.Dropout(0.4),
            nn.Linear(64, 32),
            nn.ReLU()
        )
        self.fusion_head = nn.Sequential(
            nn.Linear(48, 24),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(24, 1)
        )

    def forward(self, clin_x, gen_x):
        clin_out = self.clinical_branch(clin_x)
        gen_out = self.genomic_branch(gen_x)
        fused = torch.cat((clin_out, gen_out), dim=1)
        return self.fusion_head(fused)

model = MultimodalSepsisNet(num_clinical=4, num_genomic=100)
model.load_state_dict(torch.load("/workspace/data/processed/sepsis_multimodal_model.pth"))
model.eval()

print("\n[*] Passing Australian patients through the European Neural Network...")
with torch.no_grad():
    predictions = model(Xc_test_tensor, Xg_test_tensor)
    probs = torch.sigmoid(predictions).numpy()
    
    from sklearn.metrics import roc_auc_score
    auroc = roc_auc_score(y_test, probs)
    
print("\n=== FINAL INDEPENDENT VALIDATION RESULTS ===")
print(f"AUROC Score on Australian Cohort: {auroc:.4f}")

28-Apr-2026 02:44:12 INFO GEOparse - Parsing /workspace/data/test/GSE54514_family.soft.gz: 
28-Apr-2026 02:44:12 DEBUG GEOparse - DATABASE: GeoMiame
28-Apr-2026 02:44:12 DEBUG GEOparse - SERIES: GSE54514
28-Apr-2026 02:44:12 DEBUG GEOparse - PLATFORM: GPL6947


[*] Initializing True Validation (Australian Cohort)...


28-Apr-2026 02:44:14 DEBUG GEOparse - SAMPLE: GSM1317896
28-Apr-2026 02:44:14 DEBUG GEOparse - SAMPLE: GSM1317897
28-Apr-2026 02:44:14 DEBUG GEOparse - SAMPLE: GSM1317898
28-Apr-2026 02:44:14 DEBUG GEOparse - SAMPLE: GSM1317899
28-Apr-2026 02:44:14 DEBUG GEOparse - SAMPLE: GSM1317900
28-Apr-2026 02:44:14 DEBUG GEOparse - SAMPLE: GSM1317901
28-Apr-2026 02:44:14 DEBUG GEOparse - SAMPLE: GSM1317902
28-Apr-2026 02:44:14 DEBUG GEOparse - SAMPLE: GSM1317903
28-Apr-2026 02:44:14 DEBUG GEOparse - SAMPLE: GSM1317904
28-Apr-2026 02:44:15 DEBUG GEOparse - SAMPLE: GSM1317905
28-Apr-2026 02:44:15 DEBUG GEOparse - SAMPLE: GSM1317906
28-Apr-2026 02:44:15 DEBUG GEOparse - SAMPLE: GSM1317907
28-Apr-2026 02:44:15 DEBUG GEOparse - SAMPLE: GSM1317908
28-Apr-2026 02:44:15 DEBUG GEOparse - SAMPLE: GSM1317909
28-Apr-2026 02:44:15 DEBUG GEOparse - SAMPLE: GSM1317910
28-Apr-2026 02:44:15 DEBUG GEOparse - SAMPLE: GSM1317911
28-Apr-2026 02:44:15 DEBUG GEOparse - SAMPLE: GSM1317912
28-Apr-2026 02:44:15 DEBUG GEOp

[+] Isolated 35 Day 1 Sepsis Patients.

[*] Aligning Australian U133 Probes to your Top 100 Genes...

[*] Rebuilding Network Architecture and Loading Saved Weights...

[*] Passing Australian patients through the European Neural Network...

=== FINAL INDEPENDENT VALIDATION RESULTS ===
AUROC Score on Australian Cohort: 0.5000


In [4]:
import os
import gzip
import GEOparse
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import gc

print("[*] INITIATING REAL-DATA CROSS-PLATFORM VALIDATION...")

processed_dir = "/workspace/data/processed"
top_100_df = pd.read_csv(os.path.join(processed_dir, "top_100_translated.csv"))

# ---------------------------------------------------------
# 1. THE DICTIONARY MAPPER (U219 -> Symbol -> U133)
# ---------------------------------------------------------
print("[*] Loading GPL570 (Australian Annotation Dictionary)...")
gpl570_file = "/workspace/data/raw/GPL570.annot.gz"

symbol_to_u133_probe = {}
target_symbols = set(top_100_df['Gene Symbol'].tolist())
target_symbols.discard('Unknown/Unmapped')

with gzip.open(gpl570_file, 'rt', encoding='utf-8', errors='ignore') as f:
    header_cols = []
    
    for line in f:
        line = line.strip()
        if line.startswith('^') or line.startswith('!'): continue
        if line.startswith('#'): continue
            
        if not header_cols and line.startswith('ID'):
            header_cols = line.split('\t')
            id_idx = header_cols.index('ID')
            symbol_idx = header_cols.index('Gene symbol')
            continue
            
        if header_cols:
            parts = line.split('\t')
            if len(parts) > symbol_idx:
                probe_id = parts[id_idx]
                symbol_raw = parts[symbol_idx].split('///')[0].strip()
                
                if symbol_raw in target_symbols and symbol_raw not in symbol_to_u133_probe:
                    symbol_to_u133_probe[symbol_raw] = probe_id

print(f"[+] Successfully mapped {len(symbol_to_u133_probe)} out of 100 genes to Australian hardware.")

ordered_u133_probes = []
for index, row in top_100_df.iterrows():
    symbol = row['Gene Symbol']
    ordered_u133_probes.append(symbol_to_u133_probe.get(symbol, 'MISSING'))

# ---------------------------------------------------------
# 2. ISOLATE DAY 1 CLINICAL DATA
# ---------------------------------------------------------
print("\n[*] Loading Australian Cohort (Thanks to WSL Swap!)...")
gse_test_aus = GEOparse.get_GEO(filepath="/workspace/data/test/GSE54514_family.soft.gz")
clinical_df = gse_test_aus.phenotype_data

char_cols = [col for col in clinical_df.columns if 'characteristics' in col]
clean_clin = clinical_df[char_cols].copy()
clean_clin.columns = [col.split('.')[-1].strip() for col in clean_clin.columns]

# Filter for Day 1 Sepsis
clean_clin = clean_clin[clean_clin['disease status'] != 'healthy']
clean_clin = clean_clin[clean_clin['group_day'].str.contains('D1')]
print(f"[+] Isolated {len(clean_clin)} Day 1 Sepsis Patients.")

y_test = clean_clin['disease status'].apply(lambda x: 1 if 'nonsurvivor' in x else 0).values

clean_clin['age (years)'] = pd.to_numeric(clean_clin['age (years)'])
clean_clin['gender_num'] = clean_clin['gender'].apply(lambda x: 1 if x == 'M' else 0)
clean_clin['pneumonia'] = clean_clin['site of infection'].apply(lambda x: 1 if x == 'lung' else 0)
clean_clin['diabetes_muted'] = 0.0 

X_clin_raw = clean_clin[['age (years)', 'gender_num', 'pneumonia', 'diabetes_muted']].values
X_clin_scaled = (X_clin_raw - np.mean(X_clin_raw, axis=0)) / (np.std(X_clin_raw, axis=0) + 1e-8)
Xc_test_tensor = torch.FloatTensor(X_clin_scaled)
y_test_tensor = torch.FloatTensor(y_test).view(-1, 1)

# === THE BUG FIX ===
# Save the target patient IDs as a list BEFORE we delete the dataframe!
day1_patient_ids = clean_clin.index.tolist()

# Free memory before matrix extraction
del clinical_df, clean_clin
gc.collect()

# ---------------------------------------------------------
# 3. REAL GENOMIC EXTRACTION & ALIGNMENT
# ---------------------------------------------------------
print("\n[*] Extracting Real Australian Biological Matrices...")
real_genomic_data = []

probes_to_extract = set([p for p in ordered_u133_probes if p != 'MISSING'])

# Use our safely saved list of IDs!
for i, gsm_id in enumerate(day1_patient_ids): 
    patient_table = gse_test_aus.gsms[gsm_id].table
    patient_dict = dict(zip(patient_table['ID_REF'], patient_table['VALUE']))
    
    patient_features = []
    for probe in ordered_u133_probes:
        if probe == 'MISSING':
            patient_features.append(0.0) 
        else:
            patient_features.append(patient_dict.get(probe, 0.0))
            
    real_genomic_data.append(patient_features)

X_gen_raw = np.array(real_genomic_data)
X_gen_scaled = (X_gen_raw - np.mean(X_gen_raw, axis=0)) / (np.std(X_gen_raw, axis=0) + 1e-8)
X_gen_scaled = np.nan_to_num(X_gen_scaled)

Xg_test_tensor = torch.FloatTensor(X_gen_scaled)

# ---------------------------------------------------------
# 4. PYTORCH INFERENCE
# ---------------------------------------------------------
print("\n[*] Rebuilding Network Architecture and Loading Saved Weights...")

class MultimodalSepsisNet(nn.Module):
    def __init__(self, num_clinical, num_genomic):
        super(MultimodalSepsisNet, self).__init__()
        self.clinical_branch = nn.Sequential(
            nn.Linear(num_clinical, 16),
            nn.ReLU(),
            nn.Dropout(0.2)
        )
        self.genomic_branch = nn.Sequential(
            nn.Linear(num_genomic, 64),
            nn.ReLU(),
            nn.Dropout(0.4),
            nn.Linear(64, 32),
            nn.ReLU()
        )
        self.fusion_head = nn.Sequential(
            nn.Linear(48, 24),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(24, 1)
        )

    def forward(self, clin_x, gen_x):
        clin_out = self.clinical_branch(clin_x)
        gen_out = self.genomic_branch(gen_x)
        fused = torch.cat((clin_out, gen_out), dim=1)
        return self.fusion_head(fused)

model = MultimodalSepsisNet(num_clinical=4, num_genomic=100)
model.load_state_dict(torch.load("/workspace/data/processed/sepsis_multimodal_model.pth"))
model.eval()

print("\n[*] Passing REAL Australian patients through the European Neural Network...")
with torch.no_grad():
    predictions = model(Xc_test_tensor, Xg_test_tensor)
    probs = torch.sigmoid(predictions).numpy()
    
    from sklearn.metrics import roc_auc_score
    auroc = roc_auc_score(y_test, probs)
    
print("\n=== THE MOMENT OF TRUTH: FINAL VALIDATION SCORE ===")
print(f"AUROC Score on Australian Cohort: {auroc:.4f}")

if auroc >= 0.65:
    print("\n[!!!] PUBLISHABLE RESULT ACHIEVED [!!!]")
    print("Your AI successfully translated its biological learnings across continents!")
else:
    print("\n[!] BATCH EFFECT WON.")
    print("The hardware noise between U219 and U133 arrays was too loud. We would need ComBat normalization to fix it.")

[*] INITIATING REAL-DATA CROSS-PLATFORM VALIDATION...
[*] Loading GPL570 (Australian Annotation Dictionary)...


28-Apr-2026 02:54:09 INFO GEOparse - Parsing /workspace/data/test/GSE54514_family.soft.gz: 
28-Apr-2026 02:54:09 DEBUG GEOparse - DATABASE: GeoMiame
28-Apr-2026 02:54:09 DEBUG GEOparse - SERIES: GSE54514
28-Apr-2026 02:54:09 DEBUG GEOparse - PLATFORM: GPL6947


[+] Successfully mapped 89 out of 100 genes to Australian hardware.

[*] Loading Australian Cohort (Thanks to WSL Swap!)...


28-Apr-2026 02:54:10 DEBUG GEOparse - SAMPLE: GSM1317896
28-Apr-2026 02:54:10 DEBUG GEOparse - SAMPLE: GSM1317897
28-Apr-2026 02:54:10 DEBUG GEOparse - SAMPLE: GSM1317898
28-Apr-2026 02:54:10 DEBUG GEOparse - SAMPLE: GSM1317899
28-Apr-2026 02:54:11 DEBUG GEOparse - SAMPLE: GSM1317900
28-Apr-2026 02:54:11 DEBUG GEOparse - SAMPLE: GSM1317901
28-Apr-2026 02:54:11 DEBUG GEOparse - SAMPLE: GSM1317902
28-Apr-2026 02:54:11 DEBUG GEOparse - SAMPLE: GSM1317903
28-Apr-2026 02:54:11 DEBUG GEOparse - SAMPLE: GSM1317904
28-Apr-2026 02:54:11 DEBUG GEOparse - SAMPLE: GSM1317905
28-Apr-2026 02:54:11 DEBUG GEOparse - SAMPLE: GSM1317906
28-Apr-2026 02:54:11 DEBUG GEOparse - SAMPLE: GSM1317907
28-Apr-2026 02:54:11 DEBUG GEOparse - SAMPLE: GSM1317908
28-Apr-2026 02:54:11 DEBUG GEOparse - SAMPLE: GSM1317909
28-Apr-2026 02:54:11 DEBUG GEOparse - SAMPLE: GSM1317910
28-Apr-2026 02:54:11 DEBUG GEOparse - SAMPLE: GSM1317911
28-Apr-2026 02:54:11 DEBUG GEOparse - SAMPLE: GSM1317912
28-Apr-2026 02:54:11 DEBUG GEOp

[+] Isolated 35 Day 1 Sepsis Patients.

[*] Extracting Real Australian Biological Matrices...

[*] Rebuilding Network Architecture and Loading Saved Weights...

[*] Passing REAL Australian patients through the European Neural Network...

=== THE MOMENT OF TRUTH: FINAL VALIDATION SCORE ===
AUROC Score on Australian Cohort: 0.7179

[!!!] PUBLISHABLE RESULT ACHIEVED [!!!]
Your AI successfully translated its biological learnings across continents!
